# Phase 3 — Convergence

**Paper 1 · Unrecognized kidney and heart damage · AI-READI v3.0.0**

Phases 0–2 explored; Phase 3 converges. Three steps, in the plan's order:

| Step | Question | Output |
|---|---|---|
| **E3.1** | Which of the 207 Phase-2 associations and 11 core claims meet all four plan criteria? | `E3_1_ranking.csv`, per-site refits |
| **E3.2** | Which findings make the headline set, and exactly how are they analysed? | `PRESPEC.md`, frozen 2026-08-25 |
| **E3.3** | Do the headline analyses hold when rerun *exactly per spec*, at every site and every cutoff? | `E3_3_*.csv`, `E3.FREEZE` |

The one thing to know before reading: **Aim 2 did not survive pre-specification.** The
nerve-specific depression signal from Phase 2 attenuates under the plan's covariate set and
does not clear multiplicity correction. Aim 1 reproduces to the last decimal everywhere.

Every number here is re-read from a committed artifact through the package, never typed in.

## Setup

In [ ]:
# Thin-notebook bootstrap, same as Phases 0-2: put `src/` and the paper's `scripts/` on
# the path so this notebook calls exactly the code the runners call.
import sys, pathlib

REPO = pathlib.Path.cwd()
while not (REPO / "src" / "aireadi").exists():
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "src"), str(REPO / "papers/p1-unrecognized-damage/scripts")]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aireadi import associations, figures as fg, results, stats, thresholds
import _phase3

fg.style()
RESULTS = results.results_dir("p1")
pd.set_option("display.width", 200)
SPEC = _phase3.prespec()
print("PRESPEC", SPEC["prespec_version"], "sha256", _phase3.prespec_sha256()[:16], "…")

## E3.1 — ranking every finding on the four criteria

The plan's four criteria: effect size, survives age + severity + site, consistent across the
three sites, coherent with the core story. Three are computed; the fourth is coded from a fixed
rubric written in `run_e3_1.py`. A p-value alone does not qualify a finding.

In [ ]:
rank = pd.read_csv(RESULTS / "E3_1_ranking.csv").set_index("rank")
four = rank[rank.criteria_met == 4]
print(f"{len(rank)} Phase-2 associations scored; {int(rank.crit_survives_adjustment.sum())} survive FDR; "
      f"{len(four)} meet all four criteria")
rank.loc[four.index[:12], ["experiment", "exposure", "outcome", "estimate", "q",
                           "sites_same_direction", "sites_p_lt_05", "coherence_tier"]]

In [ ]:
# The Phase-1 core claims on the same four criteria. 'Survives adjustment' is a logistic
# model on an ordinal severity score + age + site, not the crude trend p.
core = pd.read_csv(RESULTS / "E3_1_core_claims.csv").set_index(["experiment", "claim"])
core[["spread_points", "adjusted_severity_or_per_step", "adjusted_severity_p",
      "sites_same_direction", "criteria_met"]]

### Do the candidates replicate at each site?

Same model, refitted inside UW, UAB and UCSD. The rule is direction, not per-site
significance — three sites of ~760 are individually under-powered for most of this.

In [ ]:
from IPython.display import Image
Image(filename=str(RESULTS / "E3_1_figure.png"), width=900)

## E3.2 — the headline set, frozen

`PRESPEC.md` is dated 2026-08-25 and its SHA-256 is in the log at `E3.2`. It was written
*after* the exploratory phases and the paper's Methods must say so (§7 of the spec). The
choice: Aim 1 core sweep; Aim 2 with the plan's covariate set (age + **BMI** + HbA1c +
severity + site — BMI is new relative to Phase 2); the undiagnosed-range track finding (T1);
ECG corroboration in the supplement (T2); the access-barrier negative as exploratory (T3).
Alternatives weighed are in §9 of the spec.

In [ ]:
pd.DataFrame({
    "parameter": ["cutoffs", "Aim-2 covariates", "Aim-2 family / claim rule", "T1 exposure",
                  "bootstrap", "site rule"],
    "value": [SPEC["cutoffs"], SPEC["aim2"]["covariates"],
              f"{SPEC['aim2']['fdr_family_size']} models; {SPEC['aim2']['claim_rule']}",
              f"no diabetes label & HbA1c >= {SPEC['track_undiagnosed']['hba1c_cutoff']}% (CGM mean >= {SPEC['track_undiagnosed']['cgm_mean_cutoff']:g} as replication)",
              SPEC["bootstrap"], "same direction at every site; per-site significance not required"],
}).set_index("parameter")

## E3.3 — the confirmatory reruns

### Aim 1 reproduces Phase 1 exactly — at every site and every cutoff

In [ ]:
confirm = pd.read_csv(RESULTS / "E3_3_aim1_confirmatory.csv")
assert confirm.reproduces_phase1.all()
show = confirm[confirm.stratum == "Overall"].set_index(["claim", "organ"])[["k", "n", "pct", "ci_lo", "ci_hi", "trend_z"]]
show

In [ ]:
by_site = pd.read_csv(RESULTS / "E3_3_aim1_by_site.csv")
piv = by_site.pivot_table(index=["claim", "organ"], columns="site", values="trend_z")
piv["all_same_sign"] = by_site.groupby(["claim", "organ"]).same_direction_as_pooled.all()
piv

In [ ]:
Image(filename=str(RESULTS / "E3_3_site_replication_figure.png"), width=900)

In [ ]:
# The burden across every rung of both cutoff grids: the rise with severity never goes away.
sweep = pd.read_csv(RESULTS / "E3_3_burden_sweep.csv").set_index(["organ", "cutoff"])
print("rise with severity significant at every rung:", bool((sweep.burden_trend_p < 0.05).all()))
sweep[["is_primary", "burden_pct", "burden_Healthy", "burden_Insulin", "burden_trend_z"]]

In [ ]:
Image(filename=str(RESULTS / "E3_3_burden_sweep_figure.png"), width=900)

### Aim 2 — the pre-specified test, and why it comes out differently from Phase 2

Phase 2 (age + severity + site, 16-model family) found CES-D-10 → nerve OR 1.22, q = 0.027.
Per spec (age + BMI + HbA1c + severity + site, 10-model family) it is OR 1.16 (1.02–1.32),
p = 0.025, **q = 0.245** — nominally present, not claimable. The ladder shows where it goes.

In [ ]:
aim2 = pd.read_csv(RESULTS / "E3_3_aim2_confirmatory.csv").set_index(["exposure", "outcome"])
aim2[aim2.in_corrected_family][["estimate", "ci_lo", "ci_hi", "p", "q"]].sort_values("q")

In [ ]:
ladder = pd.read_csv(RESULTS / "E3_3_aim2_ladder.csv").set_index(["step", "outcome"])
ladder.xs("abn_nerve", level="outcome")[["n", "estimate", "ci_lo", "ci_hi", "p"]]

In [ ]:
Image(filename=str(RESULTS / "E3_3_aim2_figure.png"), width=850)

In [ ]:
robust = pd.read_csv(RESULTS / "E3_3_aim2_robustness.csv").set_index(["check", "detail", "exposure", "outcome"])
robust.xs("abn_nerve", level="outcome")[["n", "estimate", "ci_lo", "ci_hi", "p"]]

### T1 — unrecognized diabetes beneath unrecognized damage: confirmed

46 participants carry no diabetes label and an HbA1c in the diabetes range. Against their
concordant peers they carry four times the odds of kidney damage; Wald and bootstrap intervals
both exclude 1, the CGM definition replicates it, and the direction holds at every site.

In [ ]:
t1 = pd.read_csv(RESULTS / "E3_3_track_undiagnosed.csv").set_index(["definition", "outcome"])
t1[["n_exposed", "events_exposed", "pct_exposed", "pct_unexposed", "estimate", "ci_lo", "ci_hi",
    "boot_ci_lo", "boot_ci_hi", "q"]]

In [ ]:
pd.read_csv(RESULTS / "E3_3_track_undiagnosed_robustness.csv").set_index(["check", "detail", "outcome"])

### T2 — the ECG agrees the troponin signal is cardiac (supplement)

In [ ]:
t2 = pd.read_csv(RESULTS / "E3_3_track_ecg.csv").set_index(["exposure", "outcome"])
t2[["n", "estimate", "ci_lo", "ci_hi", "q"]]

## Where Phase 3 leaves the paper

- **Aim 1 is frozen and bullet-proof**: 34.6% with any-organ damage; 76.6% of those with kidney
  or heart damage never told; 21.3% of all evaluable participants carrying unrecognized damage,
  40.7% on insulin — every number identical at every site's direction and every cutoff rung.
- **Aim 2 is reported as exploratory and unconfirmed**: a nerve-specific signal in Phase 2 that
  attenuated under the pre-specified adjustment and did not survive multiplicity correction. The
  spec was not changed to rescue it.
- **T1 is the one track finding promoted**, labelled exploratory-confirmatory.
- Results are frozen at `E3.FREEZE`; PRESPEC awaits Evan's sign-off.